# Part B — Neuroimaging ~ GO-i (Full Pipeline)
**Project:** Saving Brains — KMC 20-Year Follow-up  
**Module:** Part B v6 — Brain structure volumes and FA tracts vs cognitive phenotype  

---
### Pipeline overview
| Step | Description |
|---|---|
| 0 | Load data and build MRI subsample |
| 1 | Missingness audit (MCAR chi-square by GO-i) |
| 2 | Outlier audit and Winsorization (±3×IQR) |
| 3 | OLS regressions (Reg 1: GO-i \| Reg 2: KMC vs TC) |
| 4 | Sensitivity analysis (original vs Winsorized) |
| 5 | Characterisation figures (inline) |
| 6 | MLflow experiment logging |

### Dependent variables — Charpak 2022 (bilateral Left+Right)
19 structures: TotalGrayVol, CortexVol, SubCortGrayVol, Caudate, Putamen,
Pallidum, Accumbens, Amygdala, Hippocampus, Thalamus, CerebellumCortex,
CerebellumWM, CorticalWM, CorpusCallosum (Total + 5 segments)  
+ FA corticospinal tract bilateral (DTI)

### Covariates
- **Sex** — known volumetric differences
- **Fragility** (Rasch, Reg 2 only) — neonatal severity score
- **ICV** — corrected via regression residual (not as direct covariate)

### Multiple comparison correction
FDR Benjamini-Hochberg applied separately for ASEG and TRAC.  
q < 0.05 primary threshold | q < 0.10 exploratory


## 1. Environment Setup & Imports

In [0]:
"""
Suppress threadpoolctl warnings common in Databricks multi-threaded env.
Must be set before any numpy/sklearn imports.
"""
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import sys
import warnings
import logging
import tempfile
warnings.filterwarnings("ignore")

# Suppress verbose stderr during sklearn import
_stderr_orig = sys.stderr
sys.stderr = open(os.devnull, "w")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import mlflow
import mlflow.sklearn

from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests
from sklearn.linear_model import LinearRegression
import statsmodels.formula.api as smf

sys.stderr.close()
sys.stderr = _stderr_orig

sns.set_theme(style="whitegrid")
print("Imports OK ✓")


## 2. Configuration

In [0]:
# ── Data paths ────────────────────────────────────────────────────────────
INPUT_MASTER   = "/Volumes/workspace/default/kmc_data/kmc_dataset_procesado_completo.csv"
INPUT_CLUSTERS = "/Volumes/workspace/default/kmc_data/clusters_GOiv5.csv"

# Output filenames (written to temp dir and logged as MLflow artifacts)
OUT_GOI       = "partb_aseg_reg1_GOi.csv"
OUT_KMCTC     = "partb_aseg_reg2_KMCvTC.csv"
OUT_TRAC_GOI  = "partb_trac_reg1_GOi.csv"
OUT_TRAC_KTC  = "partb_trac_reg2_KMCvTC.csv"
OUT_MISSING   = "partb_missing_report.csv"
OUT_OUTLIERS  = "partb_outliers_report.csv"
OUT_SENSIB    = "partb_sensitivity_report.csv"

# ── Key column names ──────────────────────────────────────────────────────
ICV_COL   = "ASEG_EstimatedTotalIntraCranialV"
SEX_COL   = "BIRTH_sexo5"
FRAG_COL  = "FOLLOW_Fragility_Rasch_441_2PL"
GROUP_COL = "ubicac"

# ── Analysis thresholds ────────────────────────────────────────────────────
FDR_ALPHA   = 0.05   # Primary FDR threshold
FDR_EXPLORE = 0.10   # Exploratory FDR threshold
IQR_MULT    = 3.0    # Winsorization fence (±3×IQR)
WINSORIZE   = True   # True = apply | False = report only

# ── MLflow experiment ─────────────────────────────────────────────────────
# In Databricks the tracking server is pre-configured; only set the path.
MLFLOW_EXPERIMENT = "/Users/your_user@email.com/saving_brains/part_b_neuroimaging"

# ── Charpak 2022 bilateral structures ────────────────────────────────────
CHARPAK_BILATERAL = {
    "TotalGrayVol":           ["ASEG_TotalGrayVol"],
    "CortexVol":              ["ASEG_CortexVol"],
    "SubCortGrayVol":         ["ASEG_SubCortGrayVol"],
    "Caudate":                ["ASEG_LeftCaudate",               "ASEG_RightCaudate"],
    "Putamen":                ["ASEG_LeftPutamen",               "ASEG_RightPutamen"],
    "Pallidum":               ["ASEG_LeftPallidum",              "ASEG_RightPallidum"],
    "Accumbens":              ["ASEG_LeftAccumbensarea",         "ASEG_RightAccumbensarea"],
    "Amygdala":               ["ASEG_LeftAmygdala",              "ASEG_RightAmygdala"],
    "Hippocampus":            ["ASEG_LeftHippocampus",           "ASEG_RightHippocampus"],
    "Thalamus":               ["ASEG_LeftThalamusProper",        "ASEG_RightThalamusProper"],
    "CerebellumCortex":       ["ASEG_LeftCerebellumCortex",     "ASEG_RightCerebellumCortex"],
    "CerebellumWM":           ["ASEG_LeftCerebellumWhiteMatter","ASEG_RightCerebellumWhiteMatter"],
    "CorticalWM":             ["ASEG_CorticalWhiteMatterVol"],
    "CorpusCallosum_Total":   ["ASEG_CCvolumentotalc"],
    "CorpusCallosum_Ant":     ["ASEG_CC_Anterior"],
    "CorpusCallosum_CentAnt": ["ASEG_CC_Mid_Anterior"],
    "CorpusCallosum_Cent":    ["ASEG_CC_Central"],
    "CorpusCallosum_CentPos": ["ASEG_CC_Mid_Posterior"],
    "CorpusCallosum_Post":    ["ASEG_CC_Posterior"],
}

TRAC_FA_COLS = [
    "TRAC_Left_corticospinal_tract_FA_Avg",
    "TRAC_Right_corticospinal_tract_FA_Avg",
]

COVARIATES = [SEX_COL, FRAG_COL]

print("Configuration loaded ✓")


## 3. Logging & Utility Helpers

In [0]:

# Stream logs to stdout (Databricks captures cell output)
_fmt = logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s")
_ch  = logging.StreamHandler(sys.stdout)
_ch.setFormatter(_fmt)
logging.basicConfig(level=logging.INFO, handlers=[_ch], force=True)
logger = logging.getLogger("part_b_neuroimaging")


def to_num(series: pd.Series) -> pd.Series:
    """Coerce a Series to numeric; non-parseable values become NaN."""
    return pd.to_numeric(series, errors="coerce")


def section(title: str = "", width: int = 76) -> None:
    """Log a labelled section separator."""
    t = title.strip()
    if t:
        left = "─" * 4
        right = "─" * max(2, width - len(t) - 6)
        logger.info(f"\n{left} {t} {right}")
    else:
        logger.info("─" * width)

def icv_residual(sub: pd.DataFrame, vol_series: pd.Series) -> pd.Series:
    """
    Correct brain volume for head size via regression residual.

    The residual has r(residual, ICV) = 0 by construction, removing
    the confound of overall head size from volumetric comparisons.

    Parameters
    ----------
    sub        : DataFrame containing the _icv column
    vol_series : raw volume Series (bilateral sum or single structure)

    Returns
    -------
    Series of ICV-corrected residuals (same index as sub)
    """
    icv = sub["_icv"]
    ok  = vol_series.notna() & icv.notna()
    if ok.sum() < 10:
        return pd.Series(np.nan, index=sub.index)
    X   = icv[ok].values.reshape(-1, 1)
    y   = vol_series[ok].values
    reg = LinearRegression().fit(X, y)
    res = pd.Series(np.nan, index=sub.index)
    res[ok] = y - reg.predict(X)
    return res


print("Helpers defined ✓")

## 4. Step 0 — Data Loading & MRI Subsample Construction

Merges the master dataset with GO-i cluster labels and builds two
analytical subsamples:
- **sub_r1** — all MRI participants with ICV, sex, and GO-i (Reg 1)
- **sub_r2** — KMC and TC only, additionally requiring Fragility (Reg 2)

In [0]:
def load_data(master_path: str, clusters_path: str):
    """
    Load master dataset and GO-i labels; construct MRI analytical subsamples.

    Returns
    -------
    df     : full merged DataFrame
    sub    : MRI subsample (ICV not null)
    sub_r1 : Reg 1 subsample (ICV + sex + GO-i complete)
    sub_r2 : Reg 2 subsample (ICV + sex + Fragility + group in {KMC, TC})
    """
    section("STEP 0 — DATA LOADING & MRI SUBSAMPLE CONSTRUCTION")

    df       = pd.read_csv(master_path, low_memory=False)
    clusters = pd.read_csv(clusters_path)
    df       = df.merge(clusters[["code", "GO_i", "GO_i_label"]], on="code", how="inner")
    logger.info(f"Full dataset: {df.shape[0]} rows × {df.shape[1]} cols")

    # MRI subsample: participants with a valid ICV estimate
    sub = df[to_num(df[ICV_COL]).notna()].copy()
    logger.info(f"MRI subsample: n={len(sub)} (ICV not null)")

    # Convenience columns
    sub["_icv"]   = to_num(sub[ICV_COL])
    sub["_sex"]   = to_num(sub[SEX_COL])
    sub["_frag"]  = to_num(sub[FRAG_COL])
    sub["_grupo"] = to_num(sub[GROUP_COL])
    sub["_go"]    = sub["GO_i"].astype(int)

    # Reg 1: all MRI participants with complete covariates
    mask_r1 = sub[["_icv", "_sex", "_go"]].notna().all(axis=1)
    sub_r1  = sub[mask_r1].copy()

    # Reg 2: KMC vs TC only, Fragility required
    mask_r2 = (
        sub[["_icv", "_sex", "_frag", "_go"]].notna().all(axis=1)
        & sub["_grupo"].isin([1, 2])
    )
    sub_r2 = sub[mask_r2].copy()
    sub_r2["tc"] = (sub_r2["_grupo"] == 2).astype(int)

    logger.info(f"Reg 1 subsample (GO-i + sex):            n={len(sub_r1)}")
    logger.info(f"Reg 2 subsample (KMC vs TC + Fragility): n={len(sub_r2)}")

    section("GO-i distribution in MRI subsample")
    for go, label in [(1, "GO-1 High"), (2, "GO-2 Low")]:
        m = sub_r1["_go"] == go
        logger.info(
            f"  {label:<12} n={m.sum()}  "
            f"KMC={(m & (sub_r1['_grupo']==1)).sum()} "
            f"TC={(m & (sub_r1['_grupo']==2)).sum()} "
            f"Ref={(m & (sub_r1['_grupo']==3)).sum()}"
        )

    return df, sub, sub_r1, sub_r2


df, sub, sub_r1, sub_r2 = load_data(INPUT_MASTER, INPUT_CLUSTERS)

## 5. Step 1 — Missingness Audit

Checks each neuroimaging variable for missing values within the MRI
subsample and runs a chi-square MCAR test against GO-i group membership.
MRI availability itself is MNAR by protocol (only weight < 1800 g).

In [0]:
def audit_missing(sub: pd.DataFrame, sub_r1: pd.DataFrame, sub_r2: pd.DataFrame):
    """
    Audit missingness for all neuroimaging and covariate columns.

    MCAR test: chi-square between missingness indicator and GO-i group.
    p < 0.05 → MAR (missingness depends on group); otherwise MCAR assumed.

    Returns
    -------
    df_miss : DataFrame with one row per variable
    """
    section("STEP 1 — MISSINGNESS AUDIT")

    all_aseg = list({v for vl in CHARPAK_BILATERAL.values() for v in vl})
    all_cols = all_aseg + TRAC_FA_COLS + COVARIATES + [ICV_COL]

    
    logger.info(
        f"  {'Variable':<45} {'n_total':>8} {'n_miss':>8} "
        f"{'pct':>7}  {'r1_miss':>7}  {'r2_miss':>7}"
    )
    logger.info("  " + "─" * 85)

    records = []
    for col in all_cols:
        if col not in sub.columns:
            logger.warning(f"  ✗ Column not found: {col}")
            continue

        s        = to_num(sub[col])
        n_total  = len(s)
        n_miss   = int(s.isna().sum())
        pct_miss = n_miss / n_total * 100
        miss_r1  = int(to_num(sub_r1[col]).isna().sum()) if col in sub_r1.columns else np.nan
        miss_r2  = int(to_num(sub_r2[col]).isna().sum()) if col in sub_r2.columns else np.nan

        # MCAR test: does missingness differ by GO-i?
        p_mcar = np.nan
        if 0 < n_miss < n_total:
            ct = pd.crosstab(sub["_go"], s.notna().astype(int))
            if ct.shape == (2, 2):
                try:
                    _, p_mcar, _, _ = chi2_contingency(ct)
                except Exception:
                    pass

        mechanism = (
            "MAR"      if not np.isnan(p_mcar) and p_mcar < 0.05
            else "MCAR" if not np.isnan(p_mcar)
            else "MNAR/unknown"
        )
        alert = "HIGH" if pct_miss > 50 else "MEDIUM" if pct_miss > 20 else "OK"
        icon  = "🔴" if alert == "HIGH" else "🟡" if alert == "MEDIUM" else "✓"

        if n_miss > 0:
            logger.info(
                f"  {icon} {col:<43} {n_total:>8} {n_miss:>8} "
                f"{pct_miss:>6.1f}%  {miss_r1!s:>7}  {miss_r2!s:>7}"
            )

        records.append({
            "variable":    col,
            "n_total":     n_total,
            "n_missing":   n_miss,
            "pct_missing": round(pct_miss, 2),
            "miss_in_r1":  miss_r1,
            "miss_in_r2":  miss_r2,
            "p_mcar_chi2": round(float(p_mcar), 4) if not np.isnan(p_mcar) else np.nan,
            "mechanism":   mechanism,
            "alert":       alert,
        })

    df_miss = pd.DataFrame(records)

    section("Missingness summary")
    logger.info(f"  Complete variables         : {(df_miss['n_missing']==0).sum()}/{len(df_miss)}")
    logger.info(f"  Variables with any missing : {(df_miss['n_missing']>0).sum()}/{len(df_miss)}")
    logger.info(f"  Alert HIGH (>50% missing)  : {(df_miss['alert']=='HIGH').sum()}")
    logger.info(f"  Alert MEDIUM (>20% missing): {(df_miss['alert']=='MEDIUM').sum()}")
    logger.info(f"  MAR detected               : {(df_miss['mechanism']=='MAR').sum()}")
    logger.info("  Note: MRI availability is MNAR by protocol (weight < 1800 g).")

    return df_miss


df_miss = audit_missing(sub, sub_r1, sub_r2)


### Figure 1 — Missingness Heatmap by GO-i Group

In [0]:
def plot_missing_heatmap(sub: pd.DataFrame, df_miss: pd.DataFrame) -> plt.Figure:
    """Display missingness rates per variable split by GO-i group (inline only)."""
    vars_with_miss = df_miss[df_miss["n_missing"] > 0]["variable"].tolist()

    if not vars_with_miss:
        print("No missing values in MRI subsample ✓")
        return None

    rows = []
    for col in vars_with_miss:
        if col not in sub.columns:
            continue
        s = to_num(sub[col])
        short = col.replace("ASEG_","").replace("TRAC_","").replace("BIRTH_","").replace("FOLLOW_","")[:35]
        rows.append({
            "Variable":   short,
            "GO-1 High":  s[sub["_go"] == 1].isna().mean() * 100,
            "GO-2 Low":   s[sub["_go"] == 2].isna().mean() * 100,
            "MRI total":  s.isna().mean() * 100,
        })

    df_heat = pd.DataFrame(rows).set_index("Variable")
    fig, ax = plt.subplots(figsize=(8, max(3, len(rows) * 0.45)))
    sns.heatmap(
        df_heat, cmap="Reds", vmin=0, vmax=100,
        annot=True, fmt=".1f", linewidths=0.4, ax=ax,
        cbar_kws={"label": "% missing", "shrink": 0.7},
        annot_kws={"size": 8},
    )
    ax.set_title(
        "Missing values by variable and GO-i — MRI subsample (n≈299)\n"
        "(white = 0% | dark red = 100%)",
        fontweight="bold", fontsize=11,
    )
    ax.tick_params(axis="x", labelsize=10)
    ax.tick_params(axis="y", labelsize=8, rotation=0)
    plt.tight_layout()
    plt.show()  # Inline only — no file saved
    return fig


fig_missing = plot_missing_heatmap(sub, df_miss)


## 6. Step 2 — Outlier Audit & Winsorization

Sections:
- **2A** — Bilateral ASEG volumes (raw mm³)
- **2B** — ICV-corrected residuals
- **2C** — DTI FA tracts (physical range 0–1 enforced)
- **2D** — Covariates


In [0]:
def _winsorize_series(series: pd.Series, name: str):
    """
    Detect outliers via IQR fence and optionally Winsorize.

    Falls back to P2.5–P97.5 range when IQR=0 (near-constant variable).
    Returns (summary_dict, cleaned_series).
    """
    n_valid = series.notna().sum()
    if n_valid < 10:
        return None, series

    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr    = q3 - q1
    method = "IQR"

    if iqr == 0:
        p025, p975 = series.quantile(0.025), series.quantile(0.975)
        span = p975 - p025
        if span == 0:
            return {
                "variable": name, "n_outliers": 0, "pct_outliers": 0,
                "alert": "N/A", "method": "IQR=0_constant", "winsorized": False,
            }, series
        lo, hi = p025 - IQR_MULT * span * 0.5, p975 + IQR_MULT * span * 0.5
        method = "P2.5-P97.5"
    else:
        lo, hi = q1 - IQR_MULT * iqr, q3 + IQR_MULT * iqr

    outlier_mask = ((series < lo) | (series > hi)) & series.notna()
    n_out = int(outlier_mask.sum())
    pct   = n_out / n_valid * 100

    series_clean = series.copy()
    if WINSORIZE and n_out > 0:
        series_clean = series_clean.clip(lower=lo, upper=hi)

    return {
        "variable":    name,
        "n_valid":     int(n_valid),
        "Q1":          round(float(q1), 2),
        "Q3":          round(float(q3), 2),
        "IQR":         round(float(iqr), 2),
        "lower_bound": round(float(lo), 2),
        "upper_bound": round(float(hi), 2),
        "min_obs":     round(float(series.min()), 2),
        "max_obs":     round(float(series.max()), 2),
        "n_outliers":  n_out,
        "pct_outliers": round(pct, 2),
        "alert":       "HIGH" if pct > 5 else "MEDIUM" if pct > 1 else "OK",
        "method":      method,
        "winsorized":  WINSORIZE and n_out > 0,
    }, series_clean


def audit_outliers(sub, sub_r1: pd.DataFrame, sub_r2: pd.DataFrame):
    """
    Run outlier audit across all neuroimaging and covariate columns.

    Builds bilateral sums, computes ICV residuals, checks FA physical range,
    and stores Winsorized versions in sub_r1_clean / sub_r2_clean.

    Returns
    -------
    df_outliers   : audit summary DataFrame
    sub_r1_clean  : sub_r1 with cleaned bilateral and residual columns
    sub_r2_clean  : sub_r2 with cleaned bilateral columns
    """
    section(f"STEP 2 — OUTLIER AUDIT (±{IQR_MULT}×IQR | Winsorize={WINSORIZE})")

    records     = []
    sub_r1_clean = sub_r1.copy()
    sub_r2_clean = sub_r2.copy()

    # ── 2A: Raw bilateral volumes ──────────────────────────────────────
    section("2A — Raw bilateral ASEG volumes (mm³)")
    logger.info(
        f"  {'Structure':<28} {'n_out':>6} {'pct':>6}  "
        f"{'bounds':>30}  Alert"
    )
    logger.info("  " + "─" * 80)

    for name, col_list in CHARPAK_BILATERAL.items():
        cols_ok = [c for c in col_list if c in sub_r1.columns]
        if not cols_ok:
            continue
        s_r1 = (to_num(sub_r1[cols_ok[0]]) + to_num(sub_r1[cols_ok[1]]))\
               if len(cols_ok) == 2 else to_num(sub_r1[cols_ok[0]])
        s_r2 = (to_num(sub_r2[cols_ok[0]]) + to_num(sub_r2[cols_ok[1]]))\
               if len(cols_ok) == 2 else to_num(sub_r2[cols_ok[0]])

        res, s_clean_r1 = _winsorize_series(s_r1, f"{name}_bilateral")
        _,   s_clean_r2 = _winsorize_series(s_r2, f"{name}_bilateral")

        if res:
            res["section"] = "ASEG_bilateral"
            records.append(res)
            if res["n_outliers"] > 0:
                icon = "🔴" if res["alert"]=="HIGH" else "🟡" if res["alert"]=="MEDIUM" else "✓"
                logger.info(
                    f"  {icon} {name:<28} {res['n_outliers']:>6} "
                    f"{res['pct_outliers']:>5.1f}%  "
                    f"[{res['lower_bound']:>12.1f},{res['upper_bound']:>12.1f}]  "
                    f"{res['alert']}"
                )

        sub_r1_clean[f"_bil_{name}"] = s_clean_r1
        sub_r2_clean[f"_bil_{name}"] = s_clean_r2

    # ── 2B: ICV-corrected residuals ────────────────────────────────────
    # Outlier in residual = volumetric outlier not explained by head size
    section("2B — ICV-corrected residuals")
    logger.info(
        f"  {'Structure':<28} {'n_out':>6} {'pct':>6}  "
        f"{'bounds':>30}  Alert"
    )
    logger.info("  " + "─" * 80)

    for name in CHARPAK_BILATERAL:
        col_bil = f"_bil_{name}"
        if col_bil not in sub_r1_clean.columns:
            continue
        residual      = icv_residual(sub_r1_clean, sub_r1_clean[col_bil])
        res, res_clean = _winsorize_series(residual, f"{name}_residual")

        if res:
            res["section"] = "ASEG_residual"
            records.append(res)
            if res["n_outliers"] > 0:
                icon = "🔴" if res["alert"]=="HIGH" else "🟡" if res["alert"]=="MEDIUM" else "✓"
                logger.info(
                    f"  {icon} {name}_residual{'':<12} {res['n_outliers']:>6} "
                    f"{res['pct_outliers']:>5.1f}%  "
                    f"[{res['lower_bound']:>12.1f},{res['upper_bound']:>12.1f}]  "
                    f"{res['alert']}"
                )
        sub_r1_clean[f"_res_{name}"] = res_clean

    # ── 2C: DTI FA tracts ─────────────────────────────────────────────
    # FA must be in [0, 1]; values outside this range are physiologically impossible
    section("2C — DTI FA tracts (physical range [0, 1])")
    for col in TRAC_FA_COLS:
        if col not in sub_r1.columns:
            continue
        s_fa   = to_num(sub_r1[col])
        oob    = ((s_fa < 0) | (s_fa > 1)) & s_fa.notna()
        if oob.sum() > 0:
            logger.warning(f"  ⚠ {col}: {oob.sum()} values outside [0,1] → set to NaN")
            s_fa = s_fa.where(~oob, np.nan)
        low_fa = (s_fa < 0.10) & s_fa.notna()
        if low_fa.sum() > 0:
            logger.warning(f"  ⚠ {col}: {low_fa.sum()} values FA<0.10 — review DTI QC")

        res, s_clean = _winsorize_series(s_fa, col)
        if res:
            res["section"] = "TRAC"
            records.append(res)
            if res["n_outliers"] > 0:
                icon = "🔴" if res["alert"]=="HIGH" else "🟡" if res["alert"]=="MEDIUM" else "✓"
                logger.info(
                    f"  {icon} {col:<45} n_out={res['n_outliers']} "
                    f"({res['pct_outliers']:.1f}%)  {res['alert']}"
                )
        sub_r1_clean[f"_fa_{col}"] = s_clean

    # ── 2D: Covariates ────────────────────────────────────────────────
    section("2D — Covariates")
    for col in COVARIATES:
        if col not in sub_r1.columns:
            continue
        res, _ = _winsorize_series(to_num(sub_r1[col]), col)
        if res:
            res["section"] = "COV"
            records.append(res)
            logger.info(f"  ✓ {col:<43} n_out={res['n_outliers']} ({res['pct_outliers']:.1f}%)")

    # Summary by section
    df_outliers = pd.DataFrame(records)
    section("Outlier summary by section")
    for sec in ["ASEG_bilateral", "ASEG_residual", "TRAC", "COV"]:
        s = df_outliers[df_outliers.get("section", pd.Series()) == sec] \
            if "section" in df_outliers.columns else pd.DataFrame()
        if s.empty:
            continue
        logger.info(
            f"  {sec:<22} vars_with_outliers={( s['n_outliers']>0).sum()}/{len(s)}  "
            f"total_outliers={s['n_outliers'].sum()}  "
            f"alert_HIGH={(s['alert']=='HIGH').sum()}"
        )

    return df_outliers, sub_r1_clean, sub_r2_clean


df_outliers, sub_r1_clean, sub_r2_clean = audit_outliers(sub, sub_r1, sub_r2)


### Figure 2 — Outlier Distribution

In [0]:
def plot_outliers(df_outliers: pd.DataFrame, sub_r1: pd.DataFrame) -> plt.Figure:
    """Two-panel figure: outlier % bar chart and Hippocampus vs ICV scatter."""
    df_plot = df_outliers[df_outliers["n_outliers"] > 0].copy() \
              if "section" in df_outliers.columns else pd.DataFrame()

    if df_plot.empty:
        print("No outliers detected ✓")
        return None

    COLOR_MAP = {
        "ASEG_bilateral": "#2E86AB",
        "ASEG_residual":  "#F39C12",
        "TRAC":           "#C73E1D",
        "COV":            "#888888",
    }
    bar_colors = [COLOR_MAP.get(r["section"], "#888") for _, r in df_plot.iterrows()]
    labels     = df_plot["variable"].str.replace("_bilateral","",regex=False)\
                                    .str.replace("_residual"," (res)",regex=False)\
                                    .str[:35]

    fig, axes = plt.subplots(1, 2, figsize=(16, max(4, len(df_plot) * 0.4 + 2)))

    # Panel A: horizontal bar chart of outlier %
    axes[0].barh(labels, df_plot["pct_outliers"], color=bar_colors, alpha=0.82, edgecolor="white")
    axes[0].axvline(1.0, color="orange", ls="--", lw=1.5, label="1% (MEDIUM)")
    axes[0].axvline(5.0, color="red",    ls="--", lw=1.5, label="5% (HIGH)")
    axes[0].set_xlabel("% participants flagged as outliers", fontsize=10)
    axes[0].set_title(
        f"Outliers by variable\n(±{IQR_MULT}×IQR, n={len(sub_r1)})",
        fontweight="bold", fontsize=11,
    )
    legend_handles = (
        [plt.Rectangle((0,0), 1, 1, color=c, alpha=0.82) for c in COLOR_MAP.values()]
        + [plt.Line2D([0],[0], color="orange", ls="--", lw=1.5),
           plt.Line2D([0],[0], color="red",    ls="--", lw=1.5)]
    )
    axes[0].legend(legend_handles, list(COLOR_MAP.keys()) + ["1% threshold", "5% threshold"],
                   fontsize=8, loc="lower right")
    axes[0].invert_yaxis()

    # Panel B: Hippocampus bilateral vs ICV (illustrative)
    hipp_cols = ["ASEG_LeftHippocampus", "ASEG_RightHippocampus"]
    if all(c in sub_r1.columns for c in hipp_cols):
        s_hipp = to_num(sub_r1[hipp_cols[0]]) + to_num(sub_r1[hipp_cols[1]])
        s_icv  = to_num(sub_r1[ICV_COL])
        ok     = s_hipp.notna() & s_icv.notna()
        q1h, q3h = s_hipp.quantile(0.25), s_hipp.quantile(0.75)
        iqrh = q3h - q1h
        lo_h, hi_h = q1h - IQR_MULT * iqrh, q3h + IQR_MULT * iqrh
        is_out = (s_hipp < lo_h) | (s_hipp > hi_h)

        axes[1].scatter(s_icv[ok & ~is_out], s_hipp[ok & ~is_out],
                        c="#2E86AB", alpha=0.5, s=20, label="Normal")
        if is_out.sum() > 0:
            axes[1].scatter(s_icv[ok & is_out], s_hipp[ok & is_out],
                            c="#C73E1D", s=60, marker="x", lw=2,
                            label=f"Outlier (n={is_out.sum()})")
        axes[1].axhline(lo_h, color="orange", ls=":", lw=1.5)
        axes[1].axhline(hi_h, color="orange", ls=":", lw=1.5, label="±3×IQR bounds")
        axes[1].set_xlabel("ICV (mm³)", fontsize=10)
        axes[1].set_ylabel("Hippocampus bilateral (mm³)", fontsize=10)
        axes[1].set_title("Illustrative: Hippocampus bilateral vs ICV",
                          fontweight="bold", fontsize=11)
        axes[1].legend(fontsize=9)

    plt.suptitle(
        f"Outlier Audit — Part B v6 | Winsorize={WINSORIZE}",
        fontweight="bold", fontsize=12,
    )
    plt.tight_layout()
    plt.show()  # Inline only — no file saved
    return fig


fig_outliers = plot_outliers(df_outliers, sub_r1)


## 7. Step 3 — OLS Regressions

**Reg 1:** `vol_residual_ICV ~ GO_i(binary) + sex` — cognitive phenotype effect  
**Reg 2:** `vol_residual_ICV ~ TC + sex + Fragility` — treatment group effect  

Both regressions are run for all 19 ASEG structures and FA tracts.
FDR correction (Benjamini-Hochberg) is applied separately for ASEG and TRAC.

In [0]:
def _extract_ols_terms(fit, term: str) -> dict:
    """Extract coefficient, SE, CI, p-value and R² for a single OLS term."""
    ci = fit.conf_int()
    return {
        "coef":  round(float(fit.params.get(term, np.nan)), 4),
        "se":    round(float(fit.bse.get(term, np.nan)), 4),
        "ci_lo": round(float(ci.loc[term, 0]) if term in ci.index else np.nan, 4),
        "ci_hi": round(float(ci.loc[term, 1]) if term in ci.index else np.nan, 4),
        "p_raw": round(float(fit.pvalues.get(term, np.nan)), 6),
        "r2":    round(float(fit.rsquared), 4),
    }


def _apply_fdr(*dfs: pd.DataFrame) -> None:
    """Add q_fdr column in-place to each DataFrame (Benjamini-Hochberg)."""
    for df in dfs:
        if not df.empty and "p_raw" in df.columns:
            _, qs, _, _ = multipletests(df["p_raw"].fillna(1), method="fdr_bh")
            df["q_fdr"] = qs.round(6)


def run_regressions(
    sub_r1_in: pd.DataFrame,
    sub_r2_in: pd.DataFrame,
    label: str = "winsorized",
):
    """
    Run OLS regressions for all ASEG structures and TRAC FA tracts.

    Parameters
    ----------
    sub_r1_in : Reg 1 subsample (GO-i effect)
    sub_r2_in : Reg 2 subsample (KMC vs TC effect)
    label     : "original" or "winsorized" — tagged in results for sensitivity analysis

    Returns
    -------
    df_aseg_r1, df_aseg_r2, df_trac_r1, df_trac_r2 : DataFrames sorted by p_raw
    """
    res_aseg_r1, res_aseg_r2 = [], []
    res_trac_r1, res_trac_r2 = [], []

    df1 = sub_r1_in.copy()
    df2 = sub_r2_in.copy()
    df1["go2"] = (df1["_go"] == 2).astype(int)  # Binary: GO-2 Low = 1
    df2["go2"] = (df2["_go"] == 2).astype(int)
    if "tc" not in df2.columns:
        df2["tc"] = (df2["_grupo"] == 2).astype(int)

    # ── ASEG structures ───────────────────────────────────────────────
    for name, col_list in CHARPAK_BILATERAL.items():
        cols_ok = [c for c in col_list if c in df1.columns]
        if not cols_ok:
            continue

        # Bilateral sum (or single column for unilateral structures)
        df1["_y_raw"] = (to_num(df1[cols_ok[0]]) + to_num(df1[cols_ok[1]]))\
                        if len(cols_ok) == 2 else to_num(df1[cols_ok[0]])
        df2["_y_raw"] = (to_num(df2[cols_ok[0]]) + to_num(df2[cols_ok[1]]))\
                        if len(cols_ok) == 2 else to_num(df2[cols_ok[0]])

        # Group means (raw scale, before ICV correction)
        m_go1 = df1.loc[df1["_go"] == 1, "_y_raw"].mean()
        m_go2 = df1.loc[df1["_go"] == 2, "_y_raw"].mean()
        m_kmc = df2.loc[df2["_grupo"] == 1, "_y_raw"].mean()
        m_tc  = df2.loc[df2["_grupo"] == 2, "_y_raw"].mean()

        # ICV correction via regression residual
        df1["_y"] = icv_residual(df1, df1["_y_raw"])
        df2["_y"] = icv_residual(df2, df2["_y_raw"])

        ok1 = df1[["_y", "_sex", "go2"]].notna().all(axis=1)
        ok2 = df2[["_y", "_sex", "_frag", "tc"]].notna().all(axis=1)

        try:
            fit1 = smf.ols("_y ~ go2 + _sex", data=df1[ok1]).fit()
            fit2 = smf.ols("_y ~ tc + _sex + _frag", data=df2[ok2]).fit()
            res_aseg_r1.append({
                **_extract_ols_terms(fit1, "go2"),
                "structure": name, "comparison": "GO-2_vs_GO-1",
                "n": int(ok1.sum()),
                "mean_go1": round(m_go1, 1), "mean_go2": round(m_go2, 1),
                "delta_pct": round((m_go2 - m_go1) / m_go1 * 100, 1) if m_go1 != 0 else np.nan,
                "label": label,
            })
            res_aseg_r2.append({
                **_extract_ols_terms(fit2, "tc"),
                "structure": name, "comparison": "TC_vs_KMC",
                "n": int(ok2.sum()),
                "mean_kmc": round(m_kmc, 1), "mean_tc": round(m_tc, 1),
                "delta_pct": round((m_tc - m_kmc) / m_kmc * 100, 1) if m_kmc != 0 else np.nan,
                "label": label,
            })
        except Exception as e:
            logger.warning(f"  ASEG regression failed for {name}: {e}")

    # ── DTI FA tracts ─────────────────────────────────────────────────
    trac_cols = sorted([
        c for c in df1.columns
        if c.startswith("TRAC_") and "_FA_Avg" in c
        and "Center" not in c and "Weight" not in c
    ])
    for col in trac_cols:
        df1["_y"] = to_num(df1[col])
        df2["_y"] = to_num(df2[col])
        ok1 = df1[["_y", "_sex", "go2"]].notna().all(axis=1)
        ok2 = df2[["_y", "_sex", "_frag", "tc"]].notna().all(axis=1)
        if ok1.sum() < 10:
            continue
        tract_name = col.replace("TRAC_", "").replace("_FA_Avg", "")
        try:
            fit1 = smf.ols("_y ~ go2 + _sex", data=df1[ok1]).fit()
            fit2 = smf.ols("_y ~ tc + _sex + _frag", data=df2[ok2]).fit()
            res_trac_r1.append({
                **_extract_ols_terms(fit1, "go2"),
                "tract": tract_name, "comparison": "GO-2_vs_GO-1",
                "n": int(ok1.sum()),
                "mean_go1": round(df1.loc[df1["_go"]==1,"_y"].mean(), 5),
                "mean_go2": round(df1.loc[df1["_go"]==2,"_y"].mean(), 5),
                "label": label,
            })
            res_trac_r2.append({
                **_extract_ols_terms(fit2, "tc"),
                "tract": tract_name, "comparison": "TC_vs_KMC",
                "n": int(ok2.sum()),
                "mean_kmc": round(df2.loc[df2["_grupo"]==1,"_y"].mean(), 5),
                "mean_tc":  round(df2.loc[df2["_grupo"]==2,"_y"].mean(), 5),
                "label": label,
            })
        except Exception as e:
            logger.warning(f"  TRAC regression failed for {col}: {e}")

    df_a1 = pd.DataFrame(res_aseg_r1).sort_values("p_raw")
    df_a2 = pd.DataFrame(res_aseg_r2).sort_values("p_raw")
    df_t1 = pd.DataFrame(res_trac_r1).sort_values("p_raw")
    df_t2 = pd.DataFrame(res_trac_r2).sort_values("p_raw")
    _apply_fdr(df_a1, df_a2, df_t1, df_t2)

    return df_a1, df_a2, df_t1, df_t2


section("STEP 3 — OLS REGRESSIONS (Winsorized data)")
df_aseg_r1, df_aseg_r2, df_trac_r1, df_trac_r2 = run_regressions(
    sub_r1_clean, sub_r2_clean, label="winsorized"
)

# Log summary tables
for df_r, title in [
    (df_aseg_r1, "REG 1 ASEG — GO-2 Low vs GO-1 High"),
    (df_aseg_r2, "REG 2 ASEG — TC vs KMC + sex + Fragility"),
    (df_trac_r1, "REG 1 TRAC — GO-2 Low vs GO-1 High"),
    (df_trac_r2, "REG 2 TRAC — TC vs KMC + sex + Fragility"),
]:
    if df_r.empty:
        continue
    section(title)
    col_name = "structure" if "structure" in df_r.columns else "tract"
    n05 = (df_r["q_fdr"] < FDR_ALPHA).sum()
    n10 = (df_r["q_fdr"] < FDR_EXPLORE).sum()
    logger.info(f"  q<0.05: {n05}  |  q<0.10: {n10}  |  total: {len(df_r)}")
    logger.info(
        f"  {col_name.capitalize():<28} {'β':>10} {'SE':>8} {'p':>9} {'q':>9}  Sig"
    )
    logger.info("  " + "─" * 72)
    for _, r in df_r.iterrows():
        sig = ("***" if r["q_fdr"]<0.001 else "**" if r["q_fdr"]<0.01
               else "*" if r["q_fdr"]<0.05 else "†" if r["q_fdr"]<0.10 else "ns")
        logger.info(
            f"  {r[col_name]:<28} {r['coef']:>10.3f} {r['se']:>8.3f} "
            f"{r['p_raw']:>9.4f} {r['q_fdr']:>9.4f}  {sig}"
        )


## 8. Step 4 — Sensitivity Analysis: Original vs Winsorized

Re-runs both regressions on unmodified data and compares β coefficients
and significance against the Winsorized results.  
**Decision rule:** if Δβ < 10% and significance does not flip → Winsorization
does not alter conclusions; report Winsorized results and note robustness.

In [0]:
section("STEP 4 — SENSITIVITY ANALYSIS: ORIGINAL vs WINSORIZED")
logger.info("Re-running regressions on unmodified data for comparison.\n")

df_aseg_r1_orig, df_aseg_r2_orig, df_trac_r1_orig, df_trac_r2_orig = run_regressions(
    sub_r1, sub_r2, label="original"
)

df_sensitivity = pd.DataFrame()

if not df_aseg_r1.empty and not df_aseg_r1_orig.empty:
    section("Sensitivity — ASEG Reg 1 (GO-2 vs GO-1)")
    logger.info(
        f"  {'Structure':<25} {'β_orig':>10} {'β_winsor':>10} "
        f"{'Δβ%':>8} {'p_orig':>9} {'p_winsor':>9}  Status"
    )
    logger.info("  " + "─" * 85)

    merged = df_aseg_r1_orig.merge(
        df_aseg_r1[["structure", "coef", "p_raw", "q_fdr"]],
        on="structure", suffixes=("_orig", "_win"),
    )
    rows = []
    for _, r in merged.iterrows():
        delta_b  = abs(r["coef_win"] - r["coef_orig"]) / abs(r["coef_orig"]) * 100 \
                   if r["coef_orig"] != 0 else 0
        sig_flip = (r["p_raw_orig"] < 0.05) != (r["p_raw_win"] < 0.05)
        impact   = "HIGH" if delta_b > 20 else "MEDIUM" if delta_b > 10 else "LOW"
        icon     = "🔴" if delta_b > 20 else "🟡" if delta_b > 10 else "✓"
        status   = "⚠ SIGNIFICANCE FLIPPED" if sig_flip else "stable"
        logger.info(
            f"  {icon} {r['structure']:<25} {r['coef_orig']:>10.3f} "
            f"{r['coef_win']:>10.3f} {delta_b:>7.1f}% "
            f"{r['p_raw_orig']:>9.4f} {r['p_raw_win']:>9.4f}  {status}"
        )
        rows.append({
            "structure":       r["structure"],
            "coef_original":   r["coef_orig"],
            "coef_winsorized": r["coef_win"],
            "delta_beta_pct":  round(delta_b, 2),
            "p_original":      r["p_raw_orig"],
            "p_winsorized":    r["p_raw_win"],
            "q_winsorized":    r["q_fdr_win"],
            "significance_flipped": sig_flip,
            "impact":          impact,
        })

    df_sensitivity = pd.DataFrame(rows)
    n_stable = (df_sensitivity["impact"] == "LOW").sum()
    n_flip   = df_sensitivity["significance_flipped"].sum()
    logger.info(f"\n  Impact LOW   (Δβ<10%): {n_stable}/{len(df_sensitivity)}")
    logger.info(f"  Impact MEDIUM (Δβ 10–20%): {(df_sensitivity['impact']=='MEDIUM').sum()}/{len(df_sensitivity)}")
    logger.info(f"  Impact HIGH  (Δβ>20%): {(df_sensitivity['impact']=='HIGH').sum()}/{len(df_sensitivity)}")
    logger.info(f"  Significance flipped: {n_flip}")

    if n_flip == 0:
        logger.info("\n  → Conclusion: Winsorization does NOT change statistical significance.")
        logger.info("     Report Winsorized results; mention robustness in supplementary note.")
    else:
        changed = df_sensitivity[df_sensitivity["significance_flipped"]]["structure"].tolist()
        logger.info(f"\n  → WARNING: significance flipped for {n_flip} structure(s): {changed}")
        logger.info("     Manual review required before reporting.")


## 9. Step 5 — Results Figures

All figures rendered **inline only** — saved to a temp directory and
logged as MLflow artifacts in Step 6.

In [0]:
def plot_results(
    df_aseg_r1, df_aseg_r2, df_trac_r1, df_trac_r2,
):
    """
    Generate four inline result figures.

    Figures
    -------
    Fig 3 : Volcano — ASEG Reg 1 (GO-2 vs GO-1)
    Fig 4 : Volcano — ASEG Reg 2 (TC vs KMC)
    Fig 5a/b: % difference heatmaps for Reg 1 and Reg 2
    Fig 6a/b: Volcano — TRAC Reg 1 and Reg 2

    Returns
    -------
    list of matplotlib Figure objects (for MLflow logging)
    """
    figs = []

    # ── Fig 3: Volcano ASEG Reg 1 ─────────────────────────────────────
    if not df_aseg_r1.empty:
        df_v = df_aseg_r1.copy()
        df_v["log10p"] = -np.log10(df_v["p_raw"].clip(1e-10))
        fig3, ax = plt.subplots(figsize=(9, 6))
        sc = ax.scatter(
            df_v["coef"], df_v["log10p"],
            c=df_v["p_raw"], cmap="RdYlGn_r", vmin=0, vmax=0.6,
            s=70, alpha=0.85, edgecolors="#444", linewidths=0.4, zorder=3,
        )
        plt.colorbar(sc, ax=ax, label="p-value", shrink=0.7)
        for _, row in df_v.nsmallest(8, "p_raw").iterrows():
            ax.annotate(
                row["structure"][:22], (row["coef"], row["log10p"]),
                xytext=(6, 3), textcoords="offset points", fontsize=7.5,
                color="#222", arrowprops=dict(arrowstyle="-", color="#aaa", lw=0.5),
            )
        ax.axvline(0, color="gray", ls="--", lw=1, alpha=0.5)
        ax.axhline(-np.log10(0.05), color="#E74C3C", ls=":", lw=1.2, alpha=0.7, label="p=0.05")
        ax.axhline(-np.log10(0.10), color="#F39C12", ls=":", lw=1.0, alpha=0.6, label="p=0.10")
        n05 = (df_aseg_r1["q_fdr"] < FDR_ALPHA).sum()
        n10 = (df_aseg_r1["q_fdr"] < FDR_EXPLORE).sum()
        ax.text(0.02, 0.98, f"n={len(df_v)} structures\nq<0.05: {n05} | q<0.10: {n10}",
                transform=ax.transAxes, fontsize=8, va="top", ha="left", color="#555",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="#f9f9f9", edgecolor="#ccc"))
        ax.set_xlabel("β (ICV-residual mm³)", fontsize=11)
        ax.set_ylabel("-log₁₀(p)", fontsize=11)
        ax.set_title(
            "Volcano ASEG — Reg 1: GO-2 Low vs GO-1 High\n"
            "vol_residual_ICV ~ GO_i + sex  (Winsorized data)",
            fontweight="bold", fontsize=11,
        )
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()
        figs.append(fig3)

    # ── Fig 4: Volcano ASEG Reg 2 ─────────────────────────────────────
    if not df_aseg_r2.empty:
        df_v = df_aseg_r2.copy()
        df_v["log10p"] = -np.log10(df_v["p_raw"].clip(1e-10))
        df_v["sig_label"] = "ns"
        df_v.loc[df_v["q_fdr"] < FDR_EXPLORE, "sig_label"] = "q<0.10†"
        df_v.loc[df_v["q_fdr"] < FDR_ALPHA,   "sig_label"] = "q<0.05*"
        SIG_COLORS = {"ns": "#AAAAAA", "q<0.10†": "#A23B72", "q<0.05*": "#2E86AB"}
        SIG_SIZES  = {"ns": 30, "q<0.10†": 70, "q<0.05*": 90}
        fig4, ax = plt.subplots(figsize=(9, 6))
        for level, color in SIG_COLORS.items():
            m = df_v["sig_label"] == level
            ax.scatter(df_v.loc[m, "coef"], df_v.loc[m, "log10p"],
                       c=color, s=SIG_SIZES[level], alpha=0.8, edgecolors="none",
                       label=level, zorder=3)
        for _, row in df_v.nsmallest(8, "p_raw").iterrows():
            ax.annotate(row["structure"][:22], (row["coef"], row["log10p"]),
                        xytext=(6, 3), textcoords="offset points", fontsize=7.5, color="#222")
        ax.axvline(0, color="gray", ls="--", lw=1, alpha=0.5)
        ax.axhline(-np.log10(0.05), color="#E74C3C", ls=":", lw=1.2, alpha=0.7)
        ax.set_xlabel("β (ICV-residual mm³, TC vs KMC)", fontsize=11)
        ax.set_ylabel("-log₁₀(p)", fontsize=11)
        ax.set_title(
            "Volcano ASEG — Reg 2: TC vs KMC\n"
            "vol_residual_ICV ~ TC + sex + Fragility  (Winsorized data)",
            fontweight="bold", fontsize=11,
        )
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()
        figs.append(fig4)

    # ── Fig 5: % difference heatmaps ──────────────────────────────────
    for df_r, title, col_m1, col_m2, lbl1, lbl2 in [
        (df_aseg_r1, "Reg 1: GO-2 Low vs GO-1 High",
         "mean_go1", "mean_go2", "GO-1", "GO-2"),
        (df_aseg_r2, "Reg 2: TC vs KMC",
         "mean_kmc", "mean_tc", "KMC", "TC"),
    ]:
        if df_r.empty:
            continue
        top = df_r.sort_values("p_raw").head(20)
        structs = top["structure"].tolist()
        deltas  = [
            (r[col_m2] - r[col_m1]) / abs(r[col_m1]) * 100
            if r[col_m1] != 0 else 0
            for _, r in top.iterrows()
        ]
        annots  = [
            f"{d:+.1f}%\nq={q:.3f}"
            for d, q in zip(deltas, top["q_fdr"].tolist())
        ]
        fig5, ax = plt.subplots(figsize=(max(14, len(structs) * 0.9), 3.5))
        sns.heatmap(
            np.array(deltas).reshape(1, -1),
            cmap="RdBu_r", center=0, vmin=-15, vmax=15,
            annot=np.array(annots).reshape(1, -1), fmt="",
            linewidths=0.4, ax=ax,
            xticklabels=[s[:22] for s in structs],
            yticklabels=[f"{lbl2} vs {lbl1}"],
            cbar_kws={"label": "% difference", "shrink": 0.5},
            annot_kws={"size": 7.5},
        )
        ax.set_title(
            f"{title}\n(red={lbl1} larger | blue={lbl2} larger | sorted by p)",
            fontweight="bold", fontsize=11,
        )
        ax.tick_params(axis="x", rotation=45, labelsize=8)
        plt.tight_layout()
        plt.show()
        figs.append(fig5)

    # ── Fig 6: Volcano TRAC ───────────────────────────────────────────
    for df_t, title in [
        (df_trac_r1, "Reg 1 TRAC: GO-2 Low vs GO-1 High"),
        (df_trac_r2, "Reg 2 TRAC: TC vs KMC"),
    ]:
        if df_t.empty:
            continue
        df_v = df_t.copy()
        df_v["log10p"]   = -np.log10(df_v["p_raw"].clip(1e-10))
        df_v["sig_label"] = "ns"
        df_v.loc[df_v["q_fdr"] < FDR_EXPLORE, "sig_label"] = "q<0.10†"
        df_v.loc[df_v["q_fdr"] < FDR_ALPHA,   "sig_label"] = "q<0.05*"
        TRAC_COLORS = {"ns": "#AAAAAA", "q<0.10†": "#F39C12", "q<0.05*": "#C73E1D"}
        TRAC_SIZES  = {"ns": 30, "q<0.10†": 70, "q<0.05*": 90}
        fig6, ax = plt.subplots(figsize=(8, 5))
        for level, color in TRAC_COLORS.items():
            m = df_v["sig_label"] == level
            ax.scatter(df_v.loc[m, "coef"], df_v.loc[m, "log10p"],
                       c=color, s=TRAC_SIZES[level], alpha=0.75,
                       edgecolors="none", label=level, zorder=3)
        ax.axvline(0, color="gray", ls="--", lw=1, alpha=0.5)
        ax.axhline(-np.log10(0.05), color="#F39C12", ls=":", lw=1, alpha=0.6)
        ax.set_xlabel("FA coefficient", fontsize=10)
        ax.set_ylabel("-log₁₀(p)", fontsize=10)
        ax.set_title(f"Volcano TRAC — {title}", fontweight="bold", fontsize=11)
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()
        figs.append(fig6)

    return figs


section("STEP 5 — RESULTS FIGURES")
fig_objects = plot_results(df_aseg_r1, df_aseg_r2, df_trac_r1, df_trac_r2)


## 10. Step 6 — MLflow Experiment Tracking

Logs all parameters, regression results, sensitivity report, and figures
as a single MLflow run.

> **Databricks note:** tracking server is pre-configured — no URI setup needed.

In [0]:
def log_to_mlflow(
    df_aseg_r1, df_aseg_r2, df_trac_r1, df_trac_r2,
    df_miss, df_outliers, df_sensitivity,
    fig_missing, fig_outliers, fig_objects,
):
    """
    Log the complete Part B run to MLflow.

    Logged items
    ------------
    params  : pipeline configuration (IQR threshold, FDR alphas, n subsample)
    metrics : per-regression counts of significant structures at q<0.05 and q<0.10
    artifacts:
        - regression CSVs (4 files)
        - missing report, outlier report, sensitivity report
        - all inline figures (saved to temp, logged, not persisted to disk)
    """
    section("STEP 6 — MLFLOW LOGGING")
    mlflow.set_registry_uri("databricks-uc") # Use "databricks" instead if you are not using Unity Catalog
    mlflow.pyspark.ml.autolog(disable=True)
    mlflow.set_experiment(f"/Users/ja.gutierrezb@uniandes.edu.co/pasob_neuroimagen_v7")

    with mlflow.start_run(run_name="partb_neuroimaging_v6") as run:

        # ── Parameters ────────────────────────────────────────────────
        mlflow.log_params({
            "n_subsample_r1":       len(sub_r1),
            "n_subsample_r2":       len(sub_r2),
            "n_aseg_structures":    len(CHARPAK_BILATERAL),
            "n_trac_cols":          len(TRAC_FA_COLS),
            "icv_correction":       "regression_residual",
            "outlier_method":       f"Winsorization_IQR_{IQR_MULT}x",
            "winsorize":            WINSORIZE,
            "fdr_primary":          FDR_ALPHA,
            "fdr_exploratory":      FDR_EXPLORE,
            "fdr_method":           "Benjamini-Hochberg",
            "reg1_formula":         "vol_residual_ICV ~ GO_i + sex",
            "reg2_formula":         "vol_residual_ICV ~ TC + sex + Fragility",
            "charpak_reference":    "Charpak_2022_bilateral",
        })

        # ── Metrics ───────────────────────────────────────────────────
        def _sig_counts(df, prefix):
            if df.empty or "q_fdr" not in df.columns:
                return {}
            return {
                f"{prefix}_q05": int((df["q_fdr"] < FDR_ALPHA).sum()),
                f"{prefix}_q10": int((df["q_fdr"] < FDR_EXPLORE).sum()),
                f"{prefix}_n":   len(df),
            }

        metrics = {}
        metrics.update(_sig_counts(df_aseg_r1, "aseg_r1"))
        metrics.update(_sig_counts(df_aseg_r2, "aseg_r2"))
        metrics.update(_sig_counts(df_trac_r1, "trac_r1"))
        metrics.update(_sig_counts(df_trac_r2, "trac_r2"))
        if not df_sensitivity.empty:
            metrics["sensitivity_n_flipped"] = int(df_sensitivity["significance_flipped"].sum())
            metrics["sensitivity_n_high_impact"] = int((df_sensitivity["impact"] == "HIGH").sum())
        mlflow.log_metrics(metrics)

        # ── CSV artifacts ─────────────────────────────────────────────
        with tempfile.TemporaryDirectory() as tmp:
            csv_pairs = [
                (df_aseg_r1,    OUT_GOI),
                (df_aseg_r2,    OUT_KMCTC),
                (df_trac_r1,    OUT_TRAC_GOI),
                (df_trac_r2,    OUT_TRAC_KTC),
                (df_miss,       OUT_MISSING),
                (df_outliers,   OUT_OUTLIERS),
                (df_sensitivity, OUT_SENSIB),
            ]
            for df_csv, fname in csv_pairs:
                if df_csv is not None and not df_csv.empty:
                    path = os.path.join(tmp, fname)
                    df_csv.to_csv(path, index=False)
                    mlflow.log_artifact(path, artifact_path="outputs")

            # Figures — save to temp, log, display stays inline
            fig_meta = [
                (fig_missing,  "fig_01_missing_heatmap.png"),
                (fig_outliers, "fig_02_outliers.png"),
            ]
            for i, fig in enumerate(fig_objects):
                fig_meta.append((fig, f"fig_{i+3:02d}_results.png"))

            for fig, fname in fig_meta:
                if fig is not None:
                    path = os.path.join(tmp, fname)
                    fig.savefig(path, dpi=150, bbox_inches="tight")
                    mlflow.log_artifact(path, artifact_path="figures")

            logger.info("All artifacts logged to MLflow ✓")

        run_id = run.info.run_id
        logger.info(f"\nMLflow run ID  : {run_id}")
        logger.info(f"Experiment     : {MLFLOW_EXPERIMENT}")

    return run_id


run_id = log_to_mlflow(
    df_aseg_r1, df_aseg_r2, df_trac_r1, df_trac_r2,
    df_miss, df_outliers,
    df_sensitivity if "df_sensitivity" in dir() else pd.DataFrame(),
    fig_missing, fig_outliers, fig_objects,
)
print(f"\n✓ MLflow run completed. Run ID: {run_id}")


## 11. Run Summary

In [0]:
section("RUN SUMMARY — Part B Neuroimaging v6")
logger.info(f"  MRI subsample (Reg 1)    : n={len(sub_r1)}")
logger.info(f"  KMC+TC subsample (Reg 2) : n={len(sub_r2)}")
logger.info(f"  ASEG structures          : {len(CHARPAK_BILATERAL)}")
logger.info(f"  DTI FA tracts            : {len(TRAC_FA_COLS)}")
logger.info(f"  ICV correction           : regression residual (r_ICV=0)")
logger.info(f"  Outlier treatment        : Winsorization ±{IQR_MULT}×IQR")
logger.info(f"  FDR correction           : Benjamini-Hochberg (ASEG and TRAC separately)")
logger.info(f"  MLflow run ID            : {run_id}")
logger.info("")
for df_r, name in [
    (df_aseg_r1, "ASEG Reg 1"), (df_aseg_r2, "ASEG Reg 2"),
    (df_trac_r1, "TRAC Reg 1"), (df_trac_r2, "TRAC Reg 2"),
]:
    if df_r.empty:
        continue
    n05 = (df_r["q_fdr"] < FDR_ALPHA).sum()
    n10 = (df_r["q_fdr"] < FDR_EXPLORE).sum()
    logger.info(f"  {name:<14} q<0.05={n05}  q<0.10={n10}  total={len(df_r)}")
logger.info("")
logger.info("  Next step: Part C — 20-year outcome variables comparison")
